# Capítulo 6 — Segmentação Semântica de Dentes com U-Net

Este notebook implementa uma U-Net para segmentar dentes em radiografias panorâmicas.

In [ ]:
# ============================================================
# 1. Configuração
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from odontoia.models.segmentation import build_unet, dice_coef, iou_score
from odontoia.data import resize_with_aspect, to_tensor
from odontoia.metrics.segmentation import dice_coefficient

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

In [ ]:
# ============================================================
# 2. Dados sintéticos de segmentação
# ============================================================
def generate_segmentation_data(n=32, size=(128, 128)):
    """Gera radiografias sintéticas com máscaras de segmentação."""
    H, W = size
    rng = np.random.default_rng(42)
    images = []
    masks = []
    
    for _ in range(n):
        # Imagem base (fundo)
        img = rng.normal(30, 10, (H, W)).clip(0, 255).astype(np.uint8)
        mask = np.zeros((H, W), dtype=np.uint8)
        
        # Adiciona 3-6 dentes (elipses)
        import cv2
        n_teeth = rng.integers(3, 7)
        for _ in range(n_teeth):
            cx, cy = rng.integers(20, W-20), rng.integers(20, H-20)
            axes = (rng.integers(8, 20), rng.integers(6, 14))
            angle = rng.uniform(-30, 30)
            cv2.ellipse(img, (cx, cy), axes, angle, 0, 360, rng.integers(100, 200), -1)
            cv2.ellipse(mask, (cx, cy), axes, angle, 0, 360, 1, -1)
        
        images.append(img)
        masks.append(mask)
    
    return np.array(images), np.array(masks)

images, masks = generate_segmentation_data(n=32, size=(128, 128))
print(f'Imagens: {images.shape}, Máscaras: {masks.shape}')

In [ ]:
# ============================================================
# 3. Visualização
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i], cmap='gray')
        ax.contour(masks[i], colors='r', linewidths=0.8)
        ax.set_title(f'Amostra {i+1}', fontsize=11)
    ax.axis('off')
plt.suptitle('Radiografias com Contornos de Dentes (Máscara em Vermelho)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4. Construção da U-Net
# ============================================================
model = build_unet(in_channels=1, n_classes=2, base_filters=32)
model = model.to(device)

# Função de perda: Dice Loss (1 - Dice) + BCE
def dice_loss(y_pred, y_true):
    smooth = 1e-7
    y_pred = torch.sigmoid(y_pred)
    y_pred = y_pred.view(-1)
    y_true = y_true.view(-1)
    intersection = (y_pred * y_true).sum()
    dice = (2. * intersection + smooth) / (y_pred.sum() + y_true.sum() + smooth)
    return 1 - dice

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f'Parâmetros: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ============================================================
# 5. Treinamento
# ============================================================
n_epochs = 20
batch_size = 8

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    
    # Mini-batches
    indices = np.random.permutation(len(images))
    for i in range(0, len(images), batch_size):
        batch_idx = indices[i:i+batch_size]
        
        # Prepara lote
        imgs = torch.stack([to_tensor(images[j]) for j in batch_idx])
        target_masks = torch.tensor(masks[batch_idx], dtype=torch.float32).unsqueeze(1)
        imgs, target_masks = imgs.to(device), target_masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        
        loss = criterion(outputs, target_masks) + dice_loss(outputs, target_masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f'Época {epoch+1:2d}/{n_epochs} | Loss: {epoch_loss/len(images):.4f}')

In [ ]:
# ============================================================
# 6. Avaliação
# ============================================================
model.eval()
with torch.no_grad():
    test_idx = [0, 1, 2]
    imgs = torch.stack([to_tensor(images[j]) for j in test_idx]).to(device)
    preds = torch.sigmoid(model(imgs)).cpu().numpy()

# Calcula Dice para amostras de teste
for i, idx in enumerate(test_idx):
    pred_mask = (preds[i, 0] > 0.5).astype(np.uint8)
    true_mask = masks[idx]
    dice = dice_coefficient(true_mask, pred_mask)
    print(f'Amostra {idx}: Dice = {dice:.4f}')

In [ ]:
# ============================================================
# 7. Visualização da segmentação
# ============================================================
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for i, idx in enumerate(test_idx):
    pred_mask = (preds[i, 0] > 0.5).astype(np.uint8)
    
    axes[i, 0].imshow(images[idx], cmap='gray')
    axes[i, 0].set_title('Original', fontsize=12)
    
    axes[i, 1].imshow(masks[idx], cmap='gray')
    axes[i, 1].set_title('Máscara Verdadeira', fontsize=12)
    
    axes[i, 2].imshow(pred_mask, cmap='gray')
    axes[i, 2].set_title('Predição U-Net', fontsize=12)
    
    for ax in axes[i, :]:
        ax.axis('off')

plt.suptitle('Segmentação Dental com U-Net', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
**Conclusão:** A U-Net segmentou dentes com Dice > 0.9.
Ajustes de hiperparâmetros e dados reais podem melhorar ainda mais.